In [1]:
import pymc as pm
import pytensor
import pytensor.tensor as pt
import nutpie
import arviz as az
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Simulated experiments

Simulate a process $x_i = wx_{i-1} + (1-w)\alpha, \alpha \sim \textrm{Gamma}(a, b)$

In [2]:
np.random.seed(42)
theta_sim = 10
w_sim = 0.7
a_sim = 2
b_sim = 0.2
n_steps = 250

x_sim = np.empty(n_steps + 1)
x_sim[0] = 20
alpha_sim = np.random.gamma(a_sim, 1 / b_sim, size=n_steps)

for i in range(1, n_steps + 1):
    x_sim[i] = w_sim * x_sim[i - 1] + (1 - w_sim) * alpha_sim[i - 1]

x_sim += theta_sim

In [ ]:
with pm.Model() as model:
    theta = pm.Normal("theta", mu=10, sigma=1)
    w = pm.Beta("w", alpha=2, beta=2)

    x_0 = pm.Gamma("x_0", alpha=1.5, beta=0.075)
    a = pm.Gamma("a", alpha=1.5, beta=0.075, shape=n_steps)
    alpha = (1.0 - w) * a / 0.075
    
    def step(alpha_i, x_next, w):
        return w * x_next + (1.0 - w) * alpha_i

    x, _ = pytensor.scan(
        step,
        sequences=[alpha],
        outputs_info=[x_0],
        non_sequences=[w],
    )
    
    path = theta + pt.concatenate([[x_0], x])
    pm.Deterministic("path", path)

    y = pm.StudentT("y", mu=path, sigma=1.0, nu=2, observed=x_sim)

In [4]:
pymc_model = nutpie.compile_pymc_model(model)
trace = nutpie.sample(pymc_model, draws=1000, chains=4)

In [ ]:
az.rhat(trace.posterior)

Let's try when data are a cumulative sum of this process

In [9]:
x_sim = np.empty(n_steps + 1)
x_sim[0] = 20
alpha_sim = np.random.gamma(a_sim, 1 / b_sim, size=n_steps)

for i in range(1, n_steps + 1):
    x_sim[i] = w_sim * x_sim[i - 1] + (1 - w_sim) * alpha_sim[i - 1]

x_sim = theta_sim + np.cumsum(x_sim)
x_sim_observed = np.array(x_sim, dtype=np.float64, copy=True)

In [10]:
x_sim_observed

array([  30.        ,   44.40669455,   56.50512522,   69.13687284,
         85.14498213,   97.0173245 ,  105.76744778,  114.28650181,
        120.5031339 ,  127.18613688,  135.14347074,  141.79949313,
        148.839285  ,  158.61041082,  167.05753001,  174.22799614,
        181.54573401,  187.65153058,  200.48670318,  212.04023522,
        221.50275991,  231.06421283,  240.04595534,  248.42921879,
        255.90448305,  262.67966815,  267.9863505 ,  277.89203916,
        288.6169756 ,  299.27690034,  308.99922769,  316.87344464,
        323.68814311,  329.09184847,  334.31975544,  338.91531184,
        349.52922113,  361.58438014,  372.67582738,  381.61538172,
        388.85436807,  396.07088692,  405.8790766 ,  413.39676967,
        422.4154854 ,  430.16843731,  437.26586074,  444.82920096,
        451.33495979,  457.95423105,  463.70989859,  469.28002873,
        477.44578553,  485.8591471 ,  496.00410194,  504.52942007,
        514.27288832,  521.58051692,  529.2901339 ,  538.63860

In [95]:
with pm.Model() as model:
    theta = pm.Normal("theta", mu=10, sigma=1)
    w = pm.Beta("w", alpha=2, beta=2)

    x_0 = pm.Gamma("x_0", alpha=1.5, beta=0.075)
    a = pm.Gamma("a", alpha=1.5, beta=0.075, shape=n_steps)
    alpha = (1.0 - w) * a / 0.075
    
    def step(alpha_i, x_next, w):
        return w * x_next + (1.0 - w) * alpha_i

    x, _ = pytensor.scan(
        step,
        sequences=[alpha],
        outputs_info=[x_0],
        non_sequences=[w],
    )
    
    path = theta + pt.concatenate([[x_0], x])
    path_cumsum = pt.cumsum(path)
    pm.Deterministic("path", path)

    y = pm.StudentT("y", mu=path_cumsum, sigma=1.0, nu=2, observed=x_sim_observed)

In [96]:
pymc_model = nutpie.compile_pymc_model(model)
trace = nutpie.sample(pymc_model, draws=1000, chains=4)

Progress,Draws,Divergences,Step Size,Gradients/Draw
,1400,0,0.22,31
,1400,0,0.01,1023
,1400,2,0.01,1023
,1400,8,0.01,1023


### The Data

In [ ]:
df = pd.read_csv("MSB2K/MSB2K.csv")
n_obs = len(df)
y_obs = df["age"].values
depth = df["depth"].values
errors = df["error"].values
oldest_age = y_obs.max()
youngest_age = y_obs.min()

In [ ]:
thickness = 50
core_length = 100
n_sections = int(core_length / thickness)
c = np.arange(n_sections) * thickness
d_c = thickness

# Accumulation rate prior parameters (matching R: acc.shape=1.5, acc.mean=20)
# For Gamma(alpha, beta) parameterization: mean = alpha/beta
# R uses: shape=1.5, mean=20, so beta = shape/mean = 1.5/20 = 0.075
a_a = 1.5  # Was 2 - CHANGED to match R acc.shape
b_a = 0.075  # Was 0.1 - CHANGED to match R (acc.shape/acc.mean)

# Memory prior parameters (matching R: mem.mean=0.5, mem.strength=10)
# For Beta(a, b): a = mean*strength, b = (1-mean)*strength
a_w = 5  # Was 7 - CHANGED to match R (0.5 * 10)
b_w = 5  # Was 3 - CHANGED to match R (0.5 * 10)

In [ ]:
plt.scatter(np.zeros(len(depth)), depth, marker='.', label='observations')
plt.scatter(np.zeros(len(c)), c, color="red", marker='_', label="c_i")
plt.title("Observation depths and c")
plt.legend()
plt.show()

In [ ]:
def logp_w(value):
    # p(w) = (d_s * w^(d_s/d_c - 1) / d_c) * Beta(w; a_w, b_w)
    log_b = pm.Beta.logp(value ** (1 / d_c), a_w, b_w)
    return ((1 / d_c) - 1) * pm.math.log(value) - np.log(d_c) + log_b


def random_w(rng=None, size=None):
    # w = v^d_c with v ~ Beta(a_w, b_w)
    v = rng.beta(a_w, b_w, size=size)
    return np.power(v, d_c)

coords = {"c": c[1:]}

with pm.Model(coords=coords) as model:
    w = pm.CustomDist("w", logp=logp_w, random=random_w)
    
    theta0 = pm.Uniform("theta0", lower=youngest_age-500, upper=oldest_age+500)
    
    # x = accumulation rates for each section (yr/cm)

    x_bot = pm.Gamma("x_bottom", alpha=a_a, beta=b_a)
    
    alpha = pm.Gamma("alpha", alpha=a_a, beta=b_a, shape=n_sections - 1)

    def step(alpha_i, x_next, w):
        return w * x_next + (1.0 - w) * alpha_i

    # Scan BACKWARDS: alpha is reversed, start from x_K
    x_bot_to_top, _ = pytensor.scan(
        step,
        sequences=[alpha],
        outputs_info=[x_bot],
        non_sequences=[w],
    )
    
    x = pt.concatenate([x_bot_to_top[::-1], x_bot[None]])
    pm.Deterministic("accumulation_rates", x)
    
    # theta[k] = theta0 + sum(x[i] * d_c for i=0 to k-1)
    # This is the age at each section boundary
    theta = theta0 + pt.concatenate([pt.zeros(1), pt.cumsum(x[:-1]) * d_c])
    pm.Deterministic("theta", theta)
    
    i_vals = pt.as_tensor_variable(np.arange(n_sections))
    depth_tensor = pt.as_tensor_variable(depth)
    c_tensor = pt.as_tensor_variable(c)

    i = pt.floor(depth_tensor / d_c).astype('int64')
    i = pt.clip(i, 0, n_sections - 1)
    pm.Deterministic("section_idx", i)
    
    mu = theta[i] + x[i] * (depth_tensor - c_tensor[i])
    pm.Deterministic("mu", mu)

    y = pm.StudentT("y", mu=mu, sigma=errors, nu=2, observed=y_obs)

In [145]:
# Combined reparameterization: non-centered a ~ Gamma(a_a, 1) and scale alpha = (1-w)*a/b_a
# So: a is independent of w in the prior; innovation = (1-w)*a/b_a has same marginal as before.
# This can reduce posterior correlation between w and the innovations.

with pm.Model(coords=coords) as model_reparam:
    w = pm.CustomDist("w", logp=logp_w, random=random_w)

    theta0 = pm.Uniform("theta0", lower=youngest_age - 500, upper=oldest_age + 500)

    x_bot = pm.Gamma("x_bottom", alpha=a_a, beta=b_a)

    # Non-centered: a ~ Gamma(a_a, 1). Then alpha = (1-w)*a/b_a gives innovation
    # (1-w)*alpha with alpha~Gamma(a_a,b_a), i.e. same marginal as original.
    a = pm.Gamma("a", alpha=a_a, beta=1.0, shape=n_sections - 1)
    alpha = (1.0 - w) * a / b_a
    pm.Deterministic("alpha", alpha)

    def step(alpha_i, x_next, w_val):
        return w_val * x_next + alpha_i

    x_bot_to_top, _ = pytensor.scan(
        step,
        sequences=[alpha],
        outputs_info=[x_bot],
        non_sequences=[w],
    )

    x = pt.concatenate([x_bot_to_top[::-1], x_bot[None]])
    pm.Deterministic("accumulation_rates", x)

    theta = theta0 + pt.concatenate([pt.zeros(1), pt.cumsum(x[:-1]) * d_c])
    pm.Deterministic("theta", theta)

    depth_tensor = pt.as_tensor_variable(depth)
    c_tensor = pt.as_tensor_variable(c)

    i = pt.floor(depth_tensor / d_c).astype("int64")
    i = pt.clip(i, 0, n_sections - 1)
    pm.Deterministic("section_idx", i)

    mu = theta[i] + x[i] * (depth_tensor - c_tensor[i])
    pm.Deterministic("mu", mu)

    y = pm.StudentT("y", mu=mu, sigma=errors, nu=6, observed=y_obs)

In [125]:
# Sample from the prior predictive
with model:
    prior_pred = pm.sample_prior_predictive(samples=100, random_seed=42, var_names=["w","y","theta","alpha",
    "accumulation_rates","theta","mu"])

Sampling: [alpha, theta0, w, x_bottom, y]


In [ ]:
n_draws = min(50, prior_pred.prior["mu"].draw.size)
for d in range(n_draws):
    mu_ppc = prior_pred.prior["mu"].sel(chain=0, draw=d)
    theta_ppc = prior_pred.prior["theta"].sel(chain=0, draw=d)
    plt.plot(depth, mu_ppc, color='C0', alpha=0.2)
    plt.plot(c, theta_ppc, color='C1', alpha=0.2)
plt.plot(depth, y_obs, color='k', linewidth=2, label='Observed')
plt.xlabel("Depth")
plt.ylabel("mu")
plt.legend()
plt.show()

In [81]:
pymc_model = nutpie.compile_pymc_model(model)

In [ ]:
trace = nutpie.sample(pymc_model, draws=1000, chains=4)

In [87]:
trace.posterior.mean(["chain","draw"])

<xarray.Dataset> Size: 12kB
Dimensions:        (a_dim_0: 250, a_log___dim_0: 250, path_dim_0: 251)
Coordinates:
  * a_dim_0        (a_dim_0) int64 2kB 0 1 2 3 4 5 6 ... 244 245 246 247 248 249
  * a_log___dim_0  (a_log___dim_0) int64 2kB 0 1 2 3 4 5 ... 245 246 247 248 249
  * path_dim_0     (path_dim_0) int64 2kB 0 1 2 3 4 5 ... 246 247 248 249 250
Data variables:
    a              (a_dim_0) float64 2kB 1.679 1.141 1.029 ... 2.645 3.679 2.66
    a_log__        (a_log___dim_0) float64 2kB 0.3659 -0.06326 ... 1.236 0.8228
    path           (path_dim_0) float64 2kB 30.31 27.25 24.44 ... 28.4 27.27
    theta          float64 8B 13.54
    w              float64 8B 0.6821
    w_logodds__    float64 8B 0.7642
    x_0            float64 8B 16.77
    x_0_log__      float64 8B 2.816

In [86]:
az.rhat(trace.posterior)

<xarray.Dataset> Size: 12kB
Dimensions:        (a_dim_0: 250, a_log___dim_0: 250, path_dim_0: 251)
Coordinates:
  * a_dim_0        (a_dim_0) int64 2kB 0 1 2 3 4 5 6 ... 244 245 246 247 248 249
  * a_log___dim_0  (a_log___dim_0) int64 2kB 0 1 2 3 4 5 ... 245 246 247 248 249
  * path_dim_0     (path_dim_0) int64 2kB 0 1 2 3 4 5 ... 246 247 248 249 250
Data variables:
    a              (a_dim_0) float64 2kB 1.002 1.0 1.0 1.001 ... 1.004 1.0 1.001
    a_log__        (a_log___dim_0) float64 2kB 1.002 1.0 1.0 ... 0.9999 1.0
    path           (path_dim_0) float64 2kB 1.0 1.001 1.0 ... 1.001 0.9996 1.0
    theta          float64 8B 1.002
    w              float64 8B 1.003
    w_logodds__    float64 8B 1.003
    x_0            float64 8B 1.0
    x_0_log__      float64 8B 1.0

In [ ]:
trace.posterior.mean(["chain","draw"])

<xarray.Dataset> Size: 12kB
Dimensions:        (a_dim_0: 250, a_log___dim_0: 250, path_dim_0: 251)
Coordinates:
  * a_dim_0        (a_dim_0) int64 2kB 0 1 2 3 4 5 6 ... 244 245 246 247 248 249
  * a_log___dim_0  (a_log___dim_0) int64 2kB 0 1 2 3 4 5 ... 245 246 247 248 249
  * path_dim_0     (path_dim_0) int64 2kB 0 1 2 3 4 5 ... 246 247 248 249 250
Data variables:
    a              (a_dim_0) float64 2kB 1.679 1.141 1.029 ... 2.645 3.679 2.66
    a_log__        (a_log___dim_0) float64 2kB 0.3659 -0.06326 ... 1.236 0.8228
    path           (path_dim_0) float64 2kB 30.31 27.25 24.44 ... 28.4 27.27
    theta          float64 8B 13.54
    w              float64 8B 0.6821
    w_logodds__    float64 8B 0.7642
    x_0            float64 8B 16.77
    x_0_log__      float64 8B 2.816